# Session analysis

Reads what the game logged and turns it into the numbers and figures for my
thesis. Basil Toufexis, MXEN4000 / MXEN4004, Curtin.

Pick a save from the dropdown further down, then hit **Run everything below**.
That's the whole workflow. If a section comes up empty it'll tell you why, so
you're not left wondering whether it broke.

## Setup

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")

import pandas as pd

from rehab_analysis import (
    build_catalogue, check, menu_options, prepare, use_style,
    check_selection, keep, need, StaleSelection,
    sec_calibration, sec_overview, sec_quality, sec_compare,
    sec_reaction_time, sec_accuracy, sec_force, sec_individuation,
    sec_rhythm, sec_bilateral, sec_raw, sec_onset, sec_objective_one,
    sec_exclusions, sec_phase, sec_threshold_audit, sec_cue_modality,
    sec_dose, sec_sampling_note, sec_participant_progress,
    sec_summary, write_exports,
)

use_style()

### Check the setup

Sanity check before anything touches the data. Are the packages installed, and
is the sessions folder where the notebook thinks it is? If this one complains,
fix that first, because nothing below will work until you do.

In [ ]:
ok = check()

## What is on disk

Everything recorded so far, oldest first. A **game** is one folder, one block
of one mode. Games by the same person on the same day add up to a **session**,
and those two words aren't interchangeable no matter how much they sound it.
The id on the left is what the dropdown hands to everything else.

In [ ]:
cat = build_catalogue()

if cat.empty:
    print("Nothing recorded yet, so there's nothing to list here.")
    print("Play a block in the game and run this again. Every cell below")
    print("will say it has nothing to show rather than fall over.")
else:
    print(f"{len(cat)} game(s) across {cat['session'].nunique()} session(s)")

cat[["day", "time", "who", "mode", "hand",
     "trials", "hit_rate", "status"]].rename_axis("id")

## Choose what to analyse

Click a row, then hit the button beside it. You can scope it to a single game,
or to a whole session, which means one person on one day. Pick a person instead
and you get every day they've played. Pick nothing at all and you get the
newest game.

The button runs every cell below itself, which is the entire point of it. You
can still step through by hand if you'd rather, just keep them in order.
Changing the dropdown and re-running only some of the cells used to blend two
selections into the headline table and the exported CSV without saying a word
about it. Now every cell below checks the dropdown against what actually got
loaded. Mismatch and it refuses to run, so the worst you get is a message
telling you to run them again.

In [ ]:
import ipywidgets as W
from IPython.display import display, HTML, Javascript

pick = "latest"          # what every cell below analyses

note = W.HTML("<span style='color:#64748b'>newest game until you "
              "choose</span>")
out = W.Output()         # where the button reports back

# Two Jupyters, two completely different APIs, no overlap. Classic has
# Jupyter.notebook, Lab 4 and Notebook 7 have jupyterapp.commands. If neither
# is reachable this writes into the span below instead of firing an alert()
# at you, which is what I had first and it was awful.
RUN_BELOW_JS = """
(function () {
    var say = function (text) {
        var el = document.getElementById('run-below-msg');
        if (el) { el.textContent = text; }
    };
    try {
        if (window.Jupyter && window.Jupyter.notebook) {
            window.Jupyter.notebook.execute_cells_below();
            say('Running every cell below this one.');
            return;
        }
        if (window.jupyterapp && window.jupyterapp.commands) {
            window.jupyterapp.commands.execute('notebook:run-all-below');
            say('Running every cell below this one.');
            return;
        }
        say("Can't reach the run command from this front end. "
            + "Use the menu: Run, then Run All Below.");
    } catch (err) {
        say('That did not work (' + err + '). '
            + 'Use the menu: Run, then Run All Below.');
    }
})();
"""

# Shown before the JS gets a word in. If the JS never runs at all, this is
# what you're left with, so it has to be useful on its own.
WAITING = ("<span id='run-below-msg' style='color:#64748b'>Asking Jupyter to "
           "run the cells below. If nothing happens, use Run, then Run All "
           "Below from the menu.</span>")


def chosen(change):
    global pick
    if change["new"] is None:            # a heading row, not a save
        run.disabled = True
        note.value = ("<span style='color:#b45309'>that row is a heading, "
                      "choose a save under it</span>")
        return
    pick = change["new"]
    run.disabled = False
    run.tooltip = "run every cell below this one"
    note.value = (f"<span style='color:#16a34a'>selected {pick!r}. Hit Run "
                  f"everything below, or run the cells yourself in "
                  f"order.</span>")


def run_below(_):
    # The button is disabled until something is picked, but check anyway.
    # Running the lot with no selection is the confusing case, not a
    # harmless one.
    out.clear_output()
    with out:
        if menu.value is None:
            display(HTML("<span style='color:#b45309'>pick a save "
                         "first</span>"))
            return
        display(HTML(WAITING))
        display(Javascript(RUN_BELOW_JS))    # a no-op under nbconvert


menu = W.Dropdown(options=menu_options(cat), value=None,
                  description="Analyse:", layout=W.Layout(width="660px"),
                  style={"description_width": "70px"})
run = W.Button(description="Run everything below", icon="play",
               button_style="success", disabled=True,
               tooltip="pick a save first",
               layout=W.Layout(width="210px"))
menu.observe(chosen, names="value")
run.on_click(run_below)

display(W.VBox([W.HBox([menu, run]), note, out]))

## Load the selection

Reads the trials, the metadata and the calibration once, then bolts the
calibrated force columns onto the trials. Everything below works off what this
cell builds, so it has to run again after you pick something else. That's what
the button is doing for you.

In [ ]:
ctx = prepare(pick)

cat = ctx["cat"]
sel = ctx["sel"]
folders = ctx["folders"]
metas = ctx["metas"]
sessions = ctx["sessions"]
trials = ctx["trials"]
unit = ctx["unit"]
calset = ctx["calset"]
on_task = 0.0            # seconds. The overview cell below fills this in.

# This line is the one to check if the numbers ever look like they belong to
# something else. Every cell below compares it against the dropdown.
print(f"loaded: {ctx['selection']}")

if trials.empty:
    print("\nNothing to analyse in this one. Every cell below will tell you")
    print("why it's got nothing rather than fall over, so you can keep")
    print("running them.")
else:
    print(f"{len(folders)} game(s), {len(trials)} trials, "
          f"{sel['session'].nunique()} session(s)")
    print(f"force logged in {unit}, calibration: {calset.status}")

sel[["day", "time", "who", "mode", "hand", "trials"]].rename_axis("id") \
    if not sel.empty else "nothing selected"

## Calibration this data was recorded under

What one light press was worth on each pad on the day. It's the only thing
making force comparable between fingers, so it's worth a look before you trust
anything in the force section. One block per distinct calibration, and a game
that recorded none says so plainly.

In [ ]:
check_selection(ctx, pick)
cal_tables = sec_calibration(metas, sessions, calset)

## Overview

A row per game, then time on task with the pauses taken out. The dose cell
further down reuses `on_task` from here, so if you skip this one that cell's
got nothing to work with.

In [ ]:
check_selection(ctx, pick)
on_task = keep(ctx, "on_task", sec_overview(trials, folders, metas))

## Data quality

The unglamorous checks. Cues that never reached the device, how many trials
actually carry a force reading, pauses, and any sensor drift worth a second
look.

In [ ]:
check_selection(ctx, pick)
sec_quality(trials, folders, metas)

## Comparing the games

Hit rate, speed and consistency side by side. With a single game selected
there's nothing to compare against, so it stays quiet.

In [ ]:
check_selection(ctx, pick)
comparison = sec_compare(trials)

## Reaction time

How fast they reacted, split by finger. Cued modes only (classic, adaptive and
mirror), misses dropped. The spread matters more than the average here, someone
who's all over the place is a different problem from someone who's just slow.

Rhythm is left out on purpose. Its `time_difference_ms` is a signed offset from
a beat the player already knew was coming, so it goes negative about half the
time and it isn't a reaction to anything. Same column, two different meanings
depending on the mode, and I never pool the two.

In [ ]:
check_selection(ctx, pick)
rt = keep(ctx, "rt", sec_reaction_time(trials))

## Accuracy and the challenge point

Hit rate against the 65 to 80 percent band the adaptive controller aims for.
Missed trials and wrong-finger presses get counted separately, because they're
not the same mistake.

That band belongs to the adaptive controller, so this narrows to adaptive
blocks whenever the selection has any. The cell prints its scope and the
all-cued figure right next to it. The summary at the bottom exports the
all-cued one, so when the two numbers differ you're looking at two scopes, not
a disagreement.

In [ ]:
check_selection(ctx, pick)
accuracy = keep(ctx, "accuracy", sec_accuracy(trials))

## Force

Same press, measured three different ways. Raw counts are whatever the sensor
recorded. Newtons let me check against Demouche's healthy data in absolute
terms, and the third one is that press as a fraction of the finger's own
calibration press.

Here's the part I had to accept early on. Each sensor pad reads a different
number of counts for the same real force, so raw counts aren't comparable
between fingers at all. The fraction divides the pad out, but it divides the
finger out with it, because the reference press is the patient's own light
press. A weak finger recorded a small reference, so dividing by it cancels the
weakness. Read that column as effort against the finger's own reference, never
as a strength ranking.

Newtons keep the real differences, and they keep the pad differences too. So
the cell prints both and names the limitation instead of quietly picking one.
This device can't separate the two without a known physical reference sitting
on every pad, and it hasn't got one.

In [ ]:
check_selection(ctx, pick)
force = keep(ctx, "force", sec_force(trials, unit, calset))

## Finger individuation

Target-finger force over total force, worked out on two bases.

On absolute readings it's the same basis as the enslavement figures in the
literature (13 percent unimpaired, 25.1 percent after stroke), so that's the
one to put beside them. It carries the per-pad sensitivity bias along with it.

Divide each lane by its own reference press first and the pad drops out, but
now each finger's spill is weighted by the inverse of that finger's own press
strength. That version is **not** comparable with the published figures. Both
get printed over the same trials with the basis named, so the only thing
separating them is the correction.

In [ ]:
check_selection(ctx, pick)
ind = keep(ctx, "ind", sec_individuation(trials, calset))

## Rhythm

Beat offsets, and whether the tempo was actually being tracked. Rhythm blocks
only.

In [ ]:
check_selection(ctx, pick)
rhythm = keep(ctx, "rhythm", sec_rhythm(trials))

## Both hands

Left against right, bilateral blocks only. A one-handed selection gets a plain
statement of why there's nothing here rather than a blank cell.

The reaction-time line covers cued trials with a press, nothing else. Rhythm
beat offsets and misses are excluded: pooling them gave a left mean of -95 ms
and an "asymmetry" of -8.027 on the shipped data, which turned out to be one
rhythm block's timing rather than a hand difference. Took me a while to track
that down.

Each hand gets normalised with its own calibration profile. A hand played
without one stays uncorrected and the cell names it, because the two hands sit
on eight different pads.

In [ ]:
check_selection(ctx, pick)
bilateral = keep(ctx, "bilateral", sec_bilateral(trials, unit, calset))

## Raw sample stream

The 200 Hz log behind the first selected game that has one. Press durations,
and the average shape of a press.

Most selections have no raw.csv, so this usually has nothing to show. It says
which games it looked at and why each came up empty, rather than rendering a
blank and leaving you guessing.

In [ ]:
check_selection(ctx, pick)
raw_stream = sec_raw(folders, unit, calset)

## Movement onset and rate of force development

Onset taken off the force trace rather than a threshold crossing, plus how far
apart the two reaction-time estimates end up sitting. They're two ways of
asking the same question, so the gap between them is the bit I care about.

In [ ]:
check_selection(ctx, pick)
onset = keep(ctx, "onset", sec_onset(folders, trials, unit, calset))

## Objective 1, per-finger hit rate

Each finger against the band, over its own rolling window of up to 32 trials.
The session-level figure can sit comfortably inside the band while single
fingers sit well outside it, which is the reason this section exists at all.

A finger needs 32 trials of its own before it gets one full window. Fingers
with fewer are drawn dashed and left blank in `in_band_share`, because a
rolling mean over 8 trials isn't the 32-trial block the objective is worded
over.

In [ ]:
check_selection(ctx, pick)
objective_one = sec_objective_one(trials, calset=calset)

## Trial exclusions

Trials where no cue was delivered, and presses faster than 100 ms. Headline
numbers before and after those come out.

The summary at the bottom is built from the "after exclusions" row. The
sections above print over every recorded trial unless they say otherwise, so a
figure up there can differ from the summary. This table is where you see by how
much.

In [ ]:
check_selection(ctx, pick)
flagged = keep(ctx, "flagged", sec_exclusions(trials))

## Pretest to aftertest

Stays quiet until a protocol with phases has actually been run.

In [ ]:
check_selection(ctx, pick)
phases = sec_phase(trials)

## Press thresholds in newtons

What force each finger needed before it registered a press, against the healthy
fingertip forces Demouche measured. A trigger sitting above those is a
threshold problem, not a weak finger. Easy one to misread.

In [ ]:
check_selection(ctx, pick)
thresholds = sec_threshold_audit(metas=metas, calset=calset)

## Cue modality

Visual against vibration against both. Needs blocks recorded under at least two
cue settings, otherwise there's nothing to put side by side.

In [ ]:
check_selection(ctx, pick)
cues = sec_cue_modality(trials, calset)

## Dose

Repetitions against Lang's clinical benchmark. Needs `on_task` from the
overview cell above. Without it the per-minute lines get skipped rather than
guessed at.

In [ ]:
check_selection(ctx, pick)
on_task = need(ctx, "on_task")["on_task"]
sec_dose(trials, on_task / 60)

## Sampling

How many logged samples actually carry new sensor data. That's the real
resolution behind every onset and rate-of-force figure, which is why it belongs
in the limitations.

In [ ]:
check_selection(ctx, pick)
sampling = sec_sampling_note(folders)

## Progress per participant

Every session a person has done, in order. This one covers everyone on disk
rather than just the selection, because the trend across sessions is the
outcome measure.

A **session** is one person on one day. Two blocks in one sitting are one row,
not two. Counting games as sessions turned a single sitting into a training
trend, which looked great and meant nothing.

In [ ]:
check_selection(ctx, pick)
progress = sec_participant_progress(cat=cat)

## Headline numbers

Built from the trials that can be analysed, which isn't the same as every trial
recorded. A trial whose cue command never reached the device was never
presented, and a press under 100 ms is anticipation rather than a response. The
table names the counts, so the basis is there to see.

`need` below refuses to run unless every cell above ran for the save the
dropdown currently names. That's what stops this table being stitched together
out of two different selections.

In [ ]:
check_selection(ctx, pick)

# Everything this table leans on has to have been computed for THIS
# selection. Without the check, changing the dropdown and re-running only
# some of the cells built the table out of two different saves and never
# mentioned it. Learned that one the hard way.
ran = need(ctx, "on_task", "rt", "accuracy", "force", "ind", "rhythm",
           "onset", "flagged")

summary = sec_summary(trials, unit, calset, ran["on_task"],
                      folders=folders, onset=ran["onset"],
                      accuracy=ran["accuracy"])
keep(ctx, "summary", summary)

pd.DataFrame([summary]).T.rename(columns={0: "value"})

## Save the tables

Writes the CSVs next to this notebook. Skip this cell to leave the files on
disk as they are.

`selected_trials.csv` keeps every recorded trial and tacks `excluded` and
`exclusion_reason` onto each row. Filter on `excluded == False` and you land on
the same figures as the summary. That's what those two columns are for.

In [ ]:
check_selection(ctx, pick)
ran = need(ctx, "summary", "ind")

write_exports(ran["summary"], trials, calset, ran["ind"])